# Streaming estructurado del **viento**: ventanas, watermarking, checkpoint y deduplicación
**Nayder Arce Mayta** · Proyecto ClimateRisk Andes · Producto U2 · topic `viento-eventos` (3 particiones, key = `sensorId`)

**Antes de correr:** Kafka, `uso-viento` (consumer, bridge y productor) arriba. Productor cada 15 s:
```
docker compose -f uso-viento/compose.yml exec -e INTERVALO_S=15 uso-viento python productor_openmeteo.py
```
Cada demo **arranca, espera, se detiene sola y muestra el resultado** → se puede usar *Restart & Run All* (≈ 8 min).

In [1]:
import os, time, json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, IntegerType

BASE = os.getenv("VIENTO_BASE", "/opt")
ART = f"{BASE}/artifacts"
KAFKA = os.getenv("KAFKA_BOOTSTRAP", "kafka:9092")
TOPIC = "viento-eventos"
PUSHGATEWAY = os.getenv("PUSHGATEWAY", "http://pushgateway:9091")

spark = (SparkSession.builder
         .appName("u2-streaming-viento-nayder")
         .master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4040")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "|", time.strftime("%Y-%m-%d %H:%M:%S"))

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ea8b596b-9e25-4df1-b94c-c80786d981ee;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Spark 4.2.0 | 2026-11-03 00:52:45


## 1. Lectura batch del topic (qué hay guardado en Kafka)

In [2]:
lote = (spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA)
        .option("subscribe", TOPIC).option("startingOffsets", "earliest").load())
print("mensajes en el topic:", lote.count())
lote.groupBy("partition").count().orderBy("partition").show()      # misma key → misma partición
lote.selectExpr("CAST(key AS STRING) key", "partition", "offset", "CAST(value AS STRING) value") \
    .orderBy(F.desc("offset")).show(3, truncate=80)

mensajes en el topic: 661
+---------+-----+
|partition|count|
+---------+-----+
|        0|  119|
|        1|  121|
|        2|  421|
+---------+-----+



+--------------+---------+------+--------------------------------------------------------------------------------+
|           key|partition|offset|                                                                           value|
+--------------+---------+------+--------------------------------------------------------------------------------+
|puno-viento-02|        2|   421|{"tipoEvento": "viento.lectura", "version": 1, "sensorId": "puno-viento-02", ...|
|puno-viento-02|        2|   420|{"tipoEvento": "viento.lectura", "version": 1, "sensorId": "puno-viento-02", ...|
|puno-viento-02|        2|   419|{"tipoEvento": "viento.lectura", "version": 1, "sensorId": "puno-viento-02", ...|
+--------------+---------+------+--------------------------------------------------------------------------------+
only showing top 3 rows


## 2. Stream + parseo con esquema explícito + validación (esquema vs rango)

In [3]:
# Contrato viento.lectura v1 (docs/contratos.md)
esquema_viento = (StructType()
    .add("tipoEvento", StringType()).add("version", IntegerType())
    .add("sensorId", StringType()).add("estacion", StringType())
    .add("lat", DoubleType()).add("lon", DoubleType())
    .add("velocidadViento", DoubleType()).add("direccionViento", DoubleType()).add("rafaga", DoubleType())
    .add("temperatura", DoubleType()).add("humedad", DoubleType()).add("presion", DoubleType())
    .add("obsTime", LongType()).add("origen", StringType()).add("timestamp", LongType()))

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA)
         .option("subscribe", TOPIC)
         .option("startingOffsets", "latest")
         .option("failOnDataLoss", "false")
         .load())

eventos = (crudo.select("partition", "offset", F.col("timestamp").alias("kafkaTs"),
                        F.col("value").cast("string").alias("json_str"))
           .select("partition", "offset", "kafkaTs", "json_str",
                   F.from_json("json_str", esquema_viento).alias("e"))
           .select("partition", "offset", "kafkaTs", "json_str", "e.*")
           .withColumn("ts", (F.col("timestamp") / 1000).cast("timestamp"))       # tiempo del evento (lectura)
           .withColumn("obs_ts", (F.col("obsTime") / 1000).cast("timestamp")))    # paso de 15 min del modelo
eventos.printSchema()

root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTs: timestamp (nullable = true)
 |-- json_str: string (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- version: integer (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- estacion: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocidadViento: double (nullable = true)
 |-- direccionViento: double (nullable = true)
 |-- rafaga: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- obsTime: long (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- obs_ts: timestamp (nullable = true)



In [4]:
validados = (eventos
    .withColumn("esquemaOk", F.col("sensorId").isNotNull() & F.col("velocidadViento").isNotNull()
                & F.col("rafaga").isNotNull() & F.col("obsTime").isNotNull() & F.col("timestamp").isNotNull())
    .withColumn("rangoOk", F.col("velocidadViento").between(0, 200) & F.col("rafaga").between(0, 250)
                & F.col("direccionViento").between(0, 360) & F.col("humedad").between(0, 100)))
limpios = validados.filter("esquemaOk AND rangoOk")

In [5]:
import threading, shutil
from pyspark.sql.functions import to_json, struct

def correr(df_stream, modo, segundos, nombre, mostrar=True, trigger="10 seconds"):
    """Arranca el stream, imprime cada micro-lote con datos, espera `segundos`, lo detiene
    y devuelve todas las filas emitidas (para analizarlas después)."""
    chk = f"/tmp/chk_{nombre}_{int(time.time())}"          # checkpoint nuevo en cada corrida
    filas = []
    def por_lote(df, batch_id):
        datos = df.collect()
        filas.extend(datos)
        if datos and mostrar:
            print(f"--- {nombre} | micro-lote {batch_id} | {len(datos)} filas | {time.strftime('%H:%M:%S')}")
            df.show(truncate=False)
    q = (df_stream.writeStream.outputMode(modo).foreachBatch(por_lote)
         .option("checkpointLocation", chk).trigger(processingTime=trigger).start())
    print(f">>> {nombre}: corriendo {segundos} s (modo {modo}) ...")
    q.awaitTermination(segundos)
    q.stop()
    print(f">>> {nombre}: detenido | micro-lotes: {len(q.recentProgress)} | filas emitidas: {len(filas)}")
    return filas

COLS = ["tipoEvento", "version", "sensorId", "estacion", "lat", "lon", "velocidadViento", "direccionViento", "rafaga",
        "temperatura", "humedad", "presion", "obsTime", "origen", "timestamp"]

def publicar(sensor, atraso_s, rafaga=99.0, veces=1):
    """Publica en Kafka una lectura de PRUEBA (origen = prueba-manual) con `atraso_s` segundos de atraso."""
    t = int(time.time() * 1000) - atraso_s * 1000
    df = spark.createDataFrame([("viento.lectura", 1, sensor, "Prueba", -15.5, -70.1, 50.0, 90.0, rafaga,
                                 10.0, 50.0, 640.0, t, "prueba-manual", t)], COLS)
    for _ in range(veces):
        (df.select(F.col("sensorId").alias("key"), to_json(struct(*COLS)).alias("value"))
           .write.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("topic", TOPIC).save())

def publicar_luego(segundos, *args, **kw):
    def tarea():
        time.sleep(segundos); publicar(*args, **kw)
        print(f">>> [{time.strftime('%H:%M:%S')}] publicado {args[0]} (atraso {args[1]} s, x{kw.get('veces', 1)})")
    threading.Thread(target=tarea, daemon=True).start()

### 2.1 Ver llegar las lecturas (45 s)

In [6]:
_ = correr(validados.select("partition", "offset", "sensorId", "velocidadViento", "rafaga", "obs_ts", "ts",
                            "esquemaOk", "rangoOk"), "append", 45, "consola")

>>> consola: corriendo 45 s (modo append) ...


--- consola | micro-lote 1 | 3 filas | 00:53:10
+---------+------+-----------------+---------------+------+-------------------+-----------------------+---------+-------+
|partition|offset|sensorId         |velocidadViento|rafaga|obs_ts             |ts                     |esquemaOk|rangoOk|
+---------+------+-----------------+---------------+------+-------------------+-----------------------+---------+-------+
|2        |422   |puno-viento-02   |2.1            |21.6  |2026-11-02 19:45:00|2026-11-02 19:53:06.593|true     |true   |
|1        |122   |lampa-viento-03  |2.7            |10.1  |2026-11-02 19:45:00|2026-11-02 19:53:06.593|true     |true   |
|0        |120   |juliaca-viento-01|3.2            |17.6  |2026-11-02 19:45:00|2026-11-02 19:53:06.593|true     |true   |
+---------+------+-----------------+---------------+------+-------------------+-----------------------+---------+-------+

--- consola | micro-lote 2 | 3 filas | 00:53:30


+---------+------+-----------------+---------------+------+-------------------+-----------------------+---------+-------+
|partition|offset|sensorId         |velocidadViento|rafaga|obs_ts             |ts                     |esquemaOk|rangoOk|
+---------+------+-----------------+---------------+------+-------------------+-----------------------+---------+-------+
|2        |423   |puno-viento-02   |2.1            |21.6  |2026-11-02 19:45:00|2026-11-02 19:53:22.223|true     |true   |
|1        |123   |lampa-viento-03  |2.7            |10.1  |2026-11-02 19:45:00|2026-11-02 19:53:22.223|true     |true   |
|0        |121   |juliaca-viento-01|3.2            |17.6  |2026-11-02 19:45:00|2026-11-02 19:53:22.223|true     |true   |
+---------+------+-----------------+---------------+------+-------------------+-----------------------+---------+-------+

>>> consola: detenido | micro-lotes: 3 | filas emitidas: 6


## 3. Persistencia en Parquet con **checkpoint** (tolerancia a fallos)
Este stream se queda corriendo en segundo plano hasta el final del notebook.

In [7]:
P_EVENTOS = f"{ART}/s08_viento_eventos"
CHK_EVENTOS = f"{ART}/chk_s08_eventos"
consulta_parquet = (limpios.drop("json_str").writeStream.outputMode("append").format("parquet")
                    .option("path", P_EVENTOS).option("checkpointLocation", CHK_EVENTOS)
                    .trigger(processingTime="10 seconds").start())
time.sleep(50)
g = spark.read.parquet(P_EVENTOS)
print("filas en Parquet:", g.count())
g.groupBy("sensorId").agg(F.count("*").alias("lecturas"), F.countDistinct("obsTime").alias("obsTime_distintos")).show()
print("Carpetas del checkpoint:", sorted(os.listdir(CHK_EVENTOS)))
ultimo = sorted(os.listdir(f"{CHK_EVENTOS}/offsets"), key=lambda x: int(x) if x.isdigit() else -1)[-1]
print(f"offsets/{ultimo} →", open(f"{CHK_EVENTOS}/offsets/{ultimo}").read().splitlines()[-1])

filas en Parquet: 303


+-----------------+--------+-----------------+
|         sensorId|lecturas|obsTime_distintos|
+-----------------+--------+-----------------+
|  lampa-viento-03|      99|                3|
|   puno-viento-02|      99|                3|
|juliaca-viento-01|      99|                3|
|    viento-tardio|       2|                2|
| viento-duplicado|       4|                2|
+-----------------+--------+-----------------+

Carpetas del checkpoint: ['.metadata.crc', 'commits', 'metadata', 'offsets', 'sources']
offsets/22 → {"viento-eventos":{"0":126,"1":128,"2":428}}


> **Hallazgo propio:** `lecturas` > `obsTime_distintos`: consultamos la API cada 15 s pero Open-Meteo solo
> cambia el dato cada 15 min → la misma lectura llega repetida. Se resuelve en la sección 7 (dedup).
> El archivo `offsets/N` guarda hasta qué offset de cada partición se procesó: si Spark se cae, retoma desde ahí.

## 4. Ventana **tumbling** de 1 min: sin watermark vs con watermark (60 s cada una)

In [8]:
sin_wm = (limpios.groupBy(F.window("ts", "1 minute"), "sensorId")
          .agg(F.round(F.avg("velocidadViento"), 1).alias("vel_prom"), F.max("rafaga").alias("rafaga_max"),
               F.count("*").alias("n")))
_ = correr(sin_wm, "complete", 60, "tumbling_sin_watermark", trigger="15 seconds")

>>> tumbling_sin_watermark: corriendo 60 s (modo complete) ...


--- tumbling_sin_watermark | micro-lote 1 | 3 filas | 00:54:45
+------------------------------------------+-----------------+--------+----------+---+
|window                                    |sensorId         |vel_prom|rafaga_max|n  |
+------------------------------------------+-----------------+--------+----------+---+
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|lampa-viento-03  |2.7     |10.1      |1  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|juliaca-viento-01|3.2     |17.6      |1  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|puno-viento-02   |2.1     |21.6      |1  |
+------------------------------------------+-----------------+--------+----------+---+



--- tumbling_sin_watermark | micro-lote 2 | 3 filas | 00:55:00
+------------------------------------------+-----------------+--------+----------+---+
|window                                    |sensorId         |vel_prom|rafaga_max|n  |
+------------------------------------------+-----------------+--------+----------+---+
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|lampa-viento-03  |2.7     |10.1      |2  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|juliaca-viento-01|3.2     |17.6      |2  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|puno-viento-02   |2.1     |21.6      |2  |
+------------------------------------------+-----------------+--------+----------+---+



--- tumbling_sin_watermark | micro-lote 3 | 6 filas | 00:55:15
+------------------------------------------+-----------------+--------+----------+---+
|window                                    |sensorId         |vel_prom|rafaga_max|n  |
+------------------------------------------+-----------------+--------+----------+---+
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|lampa-viento-03  |2.7     |10.1      |1  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|lampa-viento-03  |2.7     |10.1      |2  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|juliaca-viento-01|3.2     |17.6      |2  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|puno-viento-02   |2.1     |21.6      |2  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|juliaca-viento-01|3.2     |17.6      |1  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|puno-viento-02   |2.1     |21.6      |1  |
+------------------------------------------+-----------------+--------+----------+---+



--- tumbling_sin_watermark | micro-lote 4 | 6 filas | 00:55:30
+------------------------------------------+-----------------+--------+----------+---+
|window                                    |sensorId         |vel_prom|rafaga_max|n  |
+------------------------------------------+-----------------+--------+----------+---+
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|lampa-viento-03  |2.7     |10.1      |2  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|juliaca-viento-01|3.2     |17.6      |2  |
|{2026-11-02 19:54:00, 2026-11-02 19:55:00}|puno-viento-02   |2.1     |21.6      |2  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|juliaca-viento-01|3.2     |17.6      |2  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|lampa-viento-03  |2.7     |10.1      |2  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|puno-viento-02   |2.1     |21.6      |2  |
+------------------------------------------+-----------------+--------+----------+---+



>>> tumbling_sin_watermark: detenido | micro-lotes: 5 | filas emitidas: 18


Sin watermark Spark usa `complete`: **re-emite todas las ventanas** en cada micro-lote y guarda su estado para siempre (la memoria crece).

In [9]:
con_wm = (limpios.withWatermark("ts", "2 minutes")
          .groupBy(F.window("ts", "1 minute"), "sensorId")
          .agg(F.round(F.avg("velocidadViento"), 1).alias("vel_prom"), F.max("rafaga").alias("rafaga_max"),
               F.count("*").alias("n")))
_ = correr(con_wm, "update", 60, "tumbling_con_watermark", trigger="15 seconds")

>>> tumbling_con_watermark: corriendo 60 s (modo update) ...
--- tumbling_con_watermark | micro-lote 1 | 3 filas | 00:55:45
+------------------------------------------+-----------------+--------+----------+---+
|window                                    |sensorId         |vel_prom|rafaga_max|n  |
+------------------------------------------+-----------------+--------+----------+---+
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|puno-viento-02   |2.1     |21.6      |1  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|lampa-viento-03  |2.7     |10.1      |1  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|juliaca-viento-01|3.2     |17.6      |1  |
+------------------------------------------+-----------------+--------+----------+---+



--- tumbling_con_watermark | micro-lote 2 | 3 filas | 00:56:00
+------------------------------------------+-----------------+--------+----------+---+
|window                                    |sensorId         |vel_prom|rafaga_max|n  |
+------------------------------------------+-----------------+--------+----------+---+
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|puno-viento-02   |2.1     |21.6      |2  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|lampa-viento-03  |2.7     |10.1      |2  |
|{2026-11-02 19:55:00, 2026-11-02 19:56:00}|juliaca-viento-01|3.2     |17.6      |2  |
+------------------------------------------+-----------------+--------+----------+---+



--- tumbling_con_watermark | micro-lote 3 | 3 filas | 00:56:15
+------------------------------------------+-----------------+--------+----------+---+
|window                                    |sensorId         |vel_prom|rafaga_max|n  |
+------------------------------------------+-----------------+--------+----------+---+
|{2026-11-02 19:56:00, 2026-11-02 19:57:00}|juliaca-viento-01|3.2     |17.6      |1  |
|{2026-11-02 19:56:00, 2026-11-02 19:57:00}|puno-viento-02   |2.1     |21.6      |1  |
|{2026-11-02 19:56:00, 2026-11-02 19:57:00}|lampa-viento-03  |2.7     |10.1      |1  |
+------------------------------------------+-----------------+--------+----------+---+



--- tumbling_con_watermark | micro-lote 4 | 3 filas | 00:56:30
+------------------------------------------+-----------------+--------+----------+---+
|window                                    |sensorId         |vel_prom|rafaga_max|n  |
+------------------------------------------+-----------------+--------+----------+---+
|{2026-11-02 19:56:00, 2026-11-02 19:57:00}|juliaca-viento-01|3.2     |17.6      |2  |
|{2026-11-02 19:56:00, 2026-11-02 19:57:00}|puno-viento-02   |2.1     |21.6      |2  |
|{2026-11-02 19:56:00, 2026-11-02 19:57:00}|lampa-viento-03  |2.7     |10.1      |2  |
+------------------------------------------+-----------------+--------+----------+---+



>>> tumbling_con_watermark: detenido | micro-lotes: 5 | filas emitidas: 12


Con watermark (`update`): solo se emiten las ventanas que **cambiaron**, y las que quedan 2 min detrás del evento más reciente se cierran y su estado se libera.

## 5. Ventana **deslizante** (5 min, avanza cada 1 min): tendencia del viento por estación (75 s)

In [10]:
deslizante = (limpios.withWatermark("ts", "2 minutes")
              .groupBy(F.window("ts", "5 minutes", "1 minute"), "estacion")
              .agg(F.round(F.avg("velocidadViento"), 1).alias("vel_prom_5min"),
                   F.max("rafaga").alias("rafaga_max_5min"), F.count("*").alias("n")))
filas_desl = correr(deslizante, "update", 75, "deslizante", trigger="15 seconds")
print("Cada lectura cae en 5 ventanas distintas (una por minuto de solapamiento).")

>>> deslizante: corriendo 75 s (modo update) ...


--- deslizante | micro-lote 1 | 15 filas | 00:57:00
+------------------------------------------+--------+-------------+---------------+---+
|window                                    |estacion|vel_prom_5min|rafaga_max_5min|n  |
+------------------------------------------+--------+-------------+---------------+---+
|{2026-11-02 19:56:00, 2026-11-02 20:01:00}|Puno    |2.1          |21.6           |1  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Puno    |2.1          |21.6           |1  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Lampa   |2.7          |10.1           |1  |
|{2026-11-02 19:54:00, 2026-11-02 19:59:00}|Lampa   |2.7          |10.1           |1  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Juliaca |3.2          |17.6           |1  |
|{2026-11-02 19:54:00, 2026-11-02 19:59:00}|Puno    |2.1          |21.6           |1  |
|{2026-11-02 19:52:00, 2026-11-02 19:57:00}|Puno    |2.1          |21.6           |1  |
|{2026-11-02 19:56:00, 2026-11-02 20:01:00}|Lampa   |2.7          |1

--- deslizante | micro-lote 2 | 15 filas | 00:57:15
+------------------------------------------+--------+-------------+---------------+---+
|window                                    |estacion|vel_prom_5min|rafaga_max_5min|n  |
+------------------------------------------+--------+-------------+---------------+---+
|{2026-11-02 19:56:00, 2026-11-02 20:01:00}|Puno    |2.1          |21.6           |2  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Puno    |2.1          |21.6           |2  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Lampa   |2.7          |10.1           |2  |
|{2026-11-02 19:54:00, 2026-11-02 19:59:00}|Lampa   |2.7          |10.1           |2  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Juliaca |3.2          |17.6           |2  |
|{2026-11-02 19:54:00, 2026-11-02 19:59:00}|Puno    |2.1          |21.6           |2  |
|{2026-11-02 19:57:00, 2026-11-02 20:02:00}|Lampa   |2.7          |10.1           |1  |
|{2026-11-02 19:56:00, 2026-11-02 20:01:00}|Lampa   |2.7          |1

--- deslizante | micro-lote 3 | 15 filas | 00:57:30
+------------------------------------------+--------+-------------+---------------+---+
|window                                    |estacion|vel_prom_5min|rafaga_max_5min|n  |
+------------------------------------------+--------+-------------+---------------+---+
|{2026-11-02 19:56:00, 2026-11-02 20:01:00}|Puno    |2.1          |21.6           |3  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Puno    |2.1          |21.6           |3  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Lampa   |2.7          |10.1           |3  |
|{2026-11-02 19:54:00, 2026-11-02 19:59:00}|Lampa   |2.7          |10.1           |3  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Juliaca |3.2          |17.6           |3  |
|{2026-11-02 19:54:00, 2026-11-02 19:59:00}|Puno    |2.1          |21.6           |3  |
|{2026-11-02 19:57:00, 2026-11-02 20:02:00}|Lampa   |2.7          |10.1           |2  |
|{2026-11-02 19:56:00, 2026-11-02 20:01:00}|Lampa   |2.7          |1

--- deslizante | micro-lote 4 | 15 filas | 00:57:45
+------------------------------------------+--------+-------------+---------------+---+
|window                                    |estacion|vel_prom_5min|rafaga_max_5min|n  |
+------------------------------------------+--------+-------------+---------------+---+
|{2026-11-02 19:56:00, 2026-11-02 20:01:00}|Puno    |2.1          |21.6           |4  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Puno    |2.1          |21.6           |4  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Lampa   |2.7          |10.1           |4  |
|{2026-11-02 19:54:00, 2026-11-02 19:59:00}|Lampa   |2.7          |10.1           |4  |
|{2026-11-02 19:55:00, 2026-11-02 20:00:00}|Juliaca |3.2          |17.6           |4  |
|{2026-11-02 19:54:00, 2026-11-02 19:59:00}|Puno    |2.1          |21.6           |4  |
|{2026-11-02 19:57:00, 2026-11-02 20:02:00}|Lampa   |2.7          |10.1           |3  |
|{2026-11-02 19:56:00, 2026-11-02 20:01:00}|Lampa   |2.7          |1

## 6. Evento **tardío**: con watermark se descarta, sin watermark se cuenta
Se publica una lectura de prueba `viento-tardio` con **10 min de atraso**. Antes se esperan 35 s para que
lleguen lecturas reales y el watermark avance (watermark = evento más reciente − 2 min).

In [11]:
conteo_wm = limpios.withWatermark("ts", "2 minutes").groupBy(F.window("ts", "1 minute"), "sensorId").count()
publicar_luego(35, "viento-tardio", 600)
filas = correr(conteo_wm, "update", 70, "tardio_con_watermark")
n_tardio = sum(1 for r in filas if r["sensorId"] == "viento-tardio")
print(f"\nCON watermark → filas de 'viento-tardio' en el resultado: {n_tardio}  (0 = descartado por tardío)")

>>> tardio_con_watermark: corriendo 70 s (modo update) ...


--- tardio_con_watermark | micro-lote 1 | 3 filas | 00:58:10
+------------------------------------------+-----------------+-----+
|window                                    |sensorId         |count|
+------------------------------------------+-----------------+-----+
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|lampa-viento-03  |1    |
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|juliaca-viento-01|1    |
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|puno-viento-02   |1    |
+------------------------------------------+-----------------+-----+



--- tardio_con_watermark | micro-lote 2 | 3 filas | 00:58:20
+------------------------------------------+-----------------+-----+
|window                                    |sensorId         |count|
+------------------------------------------+-----------------+-----+
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|lampa-viento-03  |2    |
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|juliaca-viento-01|2    |
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|puno-viento-02   |2    |
+------------------------------------------+-----------------+-----+



>>> [00:58:25] publicado viento-tardio (atraso 600 s, x1)


--- tardio_con_watermark | micro-lote 4 | 3 filas | 00:58:40
+------------------------------------------+-----------------+-----+
|window                                    |sensorId         |count|
+------------------------------------------+-----------------+-----+
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|lampa-viento-03  |3    |
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|juliaca-viento-01|3    |
|{2026-11-02 19:58:00, 2026-11-02 19:59:00}|puno-viento-02   |3    |
+------------------------------------------+-----------------+-----+



>>> tardio_con_watermark: detenido | micro-lotes: 7 | filas emitidas: 9

CON watermark → filas de 'viento-tardio' en el resultado: 0  (0 = descartado por tardío)


In [12]:
conteo_sin = limpios.groupBy(F.window("ts", "1 minute"), "sensorId").count()
publicar_luego(20, "viento-tardio-2", 600)
filas = correr(conteo_sin, "complete", 50, "tardio_sin_watermark", mostrar=False)
n_tardio2 = sum(1 for r in filas if r["sensorId"] == "viento-tardio-2")
print(f"SIN watermark → filas de 'viento-tardio-2' en el resultado: {n_tardio2}  (> 0 = se contó aunque llegó 10 min tarde)")

>>> tardio_sin_watermark: corriendo 50 s (modo complete) ...


>>> [00:59:18] publicado viento-tardio-2 (atraso 600 s, x1)


>>> tardio_sin_watermark: detenido | micro-lotes: 5 | filas emitidas: 15
SIN watermark → filas de 'viento-tardio-2' en el resultado: 3  (> 0 = se contó aunque llegó 10 min tarde)


## 7. **Deduplicación** con watermark — la usamos de verdad: `(sensorId, obsTime)` (60 s)

In [13]:
dedup = limpios.withWatermark("obs_ts", "30 minutes").dropDuplicates(["sensorId", "obsTime"])
publicar_luego(20, "viento-duplicado", 0, rafaga=20.0, veces=2)
filas = correr(dedup.select("sensorId", "obs_ts", "velocidadViento", "rafaga", "ts"), "append", 60, "dedup")
n_dup = sum(1 for r in filas if r["sensorId"] == "viento-duplicado")
reales = [r for r in filas if r["sensorId"].endswith(("-01", "-02", "-03"))]
print(f"\n'viento-duplicado' se publicó 2 veces → aparece {n_dup} vez/veces")
print(f"Lecturas reales emitidas: {len(reales)} (una por sensor y paso de 15 min, aunque llegaron cada 15 s)")

>>> dedup: corriendo 60 s (modo append) ...


--- dedup | micro-lote 1 | 3 filas | 01:00:00
+-----------------+-------------------+---------------+------+-----------------------+
|sensorId         |obs_ts             |velocidadViento|rafaga|ts                     |
+-----------------+-------------------+---------------+------+-----------------------+
|puno-viento-02   |2026-11-02 19:45:00|2.1            |21.6  |2026-11-02 19:59:52.923|
|lampa-viento-03  |2026-11-02 19:45:00|2.7            |10.1  |2026-11-02 19:59:52.923|
|juliaca-viento-01|2026-11-02 19:45:00|3.2            |17.6  |2026-11-02 19:59:52.923|
+-----------------+-------------------+---------------+------+-----------------------+



>>> [01:00:08] publicado viento-duplicado (atraso 0 s, x2)
--- dedup | micro-lote 2 | 4 filas | 01:00:10
+-----------------+-----------------------+---------------+------+-----------------------+
|sensorId         |obs_ts                 |velocidadViento|rafaga|ts                     |
+-----------------+-----------------------+---------------+------+-----------------------+
|puno-viento-02   |2026-11-02 20:00:00    |4.8            |20.9  |2026-11-02 20:00:09.859|
|lampa-viento-03  |2026-11-02 20:00:00    |3.1            |10.8  |2026-11-02 20:00:09.859|
|viento-duplicado |2026-11-02 20:00:08.513|50.0           |20.0  |2026-11-02 20:00:08.513|
|juliaca-viento-01|2026-11-02 20:00:00    |2.2            |15.5  |2026-11-02 20:00:09.859|
+-----------------+-----------------------+---------------+------+-----------------------+



>>> dedup: detenido | micro-lotes: 5 | filas emitidas: 7

'viento-duplicado' se publicó 2 veces → aparece 1 vez/veces
Lecturas reales emitidas: 6 (una por sensor y paso de 15 min, aunque llegaron cada 15 s)


## 8. Efecto del trigger en latencia/throughput

In [14]:
def medir(intervalo, segundos=30):
    q = (limpios.writeStream.outputMode("append").format("memory")
         .queryName("m_" + intervalo.replace(" ", "_")).trigger(processingTime=intervalo).start())
    q.awaitTermination(segundos); prog = q.recentProgress; q.stop()
    lotes = [p for p in prog if p["numInputRows"] > 0]
    dur = [p["durationMs"].get("triggerExecution", 0) for p in prog]
    return {"trigger": intervalo, "microlotes": len(prog), "lotes_con_datos": len(lotes),
            "filas": sum(p["numInputRows"] for p in prog), "ms_prom_por_lote": round(sum(dur) / max(len(dur), 1), 1)}

print(medir("2 seconds")); print(medir("10 seconds"))

{'trigger': '2 seconds', 'microlotes': 4, 'lotes_con_datos': 2, 'filas': 6, 'ms_prom_por_lote': 455.8}


{'trigger': '10 seconds', 'microlotes': 3, 'lotes_con_datos': 1, 'filas': 3, 'ms_prom_por_lote': 314.3}


**Lectura:** trigger corto → más micro-lotes (menor latencia, más overhead y más archivos pequeños);
trigger largo → menos lotes más grandes (más throughput por lote, más latencia).

**Semántica de entrega:** Kafka → Spark es *at-least-once* (si se cae entre procesar y guardar el offset, se relee).
El sink Parquet + checkpoint lo vuelve *exactly-once* a nivel de archivos (offsets y archivos se confirman juntos).
La dedup por `(sensorId, obsTime)` cubre además los duplicados que **genera la propia fuente**.

### Umbrales operativos (se implementan en la observabilidad)
| Métrica | Umbral | Por qué |
|---|---|---|
| `inputRowsPerSecond` = 0 por > 5 min | alerta | el productor o la API se cayó |
| `triggerExecution` > intervalo del trigger | alerta | backpressure: Spark no alcanza |
| `avgOffsetsBehindLatest` > 100 | alerta | lag acumulado |
| % de inválidos > 5 % | alerta | cambió el contrato o la fuente |

In [15]:
consulta_parquet.stop()
for q in spark.streams.active: q.stop()
print("Streams activos:", len(spark.streams.active))

Streams activos: 0
